# RIR bank prep (IR Survey → indoor, no bathroom)

Build a reusable impulse-response bank for waveform convolution:

- Source: `training/noises/ir-survey` (McDermott IR Survey, ~270 wavs @ 32 kHz).
- **Indoor** rooms only; **exclude bathrooms/showers** and outdoor / transit spaces.
- Preprocess: mono, resample to chord sample rate (48 kHz), trim leading silence (Kusaka-style).
- Writes a single npz bank under `features/rir/` (variable-length IRs as object array).
- No chord audio here — see `mix-rir-cqt.ipynb` for convolve → CQT.

## Config

In [1]:
from os import path
import pandas as pd
from IPython.display import display

IR_SURVEY_DIR = path.normpath("../../noises/ir-survey")
FEATURES_DIR = path.normpath("../../features/rir")
BANK_TAG = "indoor_no_bathroom"
FEATURES_OUT = path.join(FEATURES_DIR, f"ir-survey-{BANK_TAG}.npz")
MANIFEST_OUT = path.join(FEATURES_DIR, f"ir-survey-{BANK_TAG}.manifest.csv")

FORCE_REBUILD = False

# Target rate for convolution with chord audio (must match CQT extract)
TARGET_SR = 48_000

# Leading-silence trim: first sample at/above peak * 10^(THRESH_DB/20)
TRIM_LEADING_DB = -40.0

# Room token = 2nd underscore field of stem (h001_Bedroom_... -> Bedroom)
# Explicit denylist: outdoor / transit / bathroom family
EXCLUDE_ROOM_TOKENS = {
    # bathroom
    "Bathroom",
    "OfficeBathroom",
    "Shower",
    # outdoor / semi-outdoor
    "Outside",
    "Outdoor",
    "OutsideStreetsOfBoston",
    "StreetsOfCambridge",
    "StreetsOfcambridge",
    "StreetsOfBoston",
    "ParkingLot",
    "Campground",
    "SuburbanBackyard",
    "PorchOfSuburbanHouse",
    "BackPorchOfSuburbanHome",
    "2ndFloorBalconyOfWoodenHouse",
    # transit / vehicle
    "Car",
    "Train",
    "TrainStation",
    "SubwayStation",
    "SwimmingPool",
}

config_df = pd.DataFrame(
    [
        {"key": "ir_survey_dir", "value": IR_SURVEY_DIR},
        {"key": "features_out", "value": FEATURES_OUT},
        {"key": "manifest_out", "value": MANIFEST_OUT},
        {"key": "bank_tag", "value": BANK_TAG},
        {"key": "force_rebuild", "value": FORCE_REBUILD},
        {"key": "target_sr", "value": TARGET_SR},
        {"key": "trim_leading_db", "value": TRIM_LEADING_DB},
        {"key": "n_exclude_tokens", "value": len(EXCLUDE_ROOM_TOKENS)},
    ]
).set_index("key")
display(config_df)

,value
key,
ir_survey_dir,../../noises/ir-survey
features_out,../../features/rir/ir-survey-indoor_no_bathroo...
manifest_out,../../features/rir/ir-survey-indoor_no_bathroo...
bank_tag,indoor_no_bathroom
force_rebuild,False
target_sr,48000
trim_leading_db,-40.0
n_exclude_tokens,20


## Select IRs

In [2]:
from pathlib import Path
import pandas as pd
from IPython.display import display


def room_token(stem: str) -> str:
    parts = stem.split("_")
    return parts[1] if len(parts) > 1 else stem


ir_paths = sorted(Path(IR_SURVEY_DIR).glob("*.wav"))
assert ir_paths, f"No wavs in {IR_SURVEY_DIR}"

rows = []
for p in ir_paths:
    room = room_token(p.stem)
    keep = room not in EXCLUDE_ROOM_TOKENS
    rows.append({
        "filename": p.name,
        "path": str(p),
        "room": room,
        "keep": int(keep),
        "reason": "" if keep else f"excluded:{room}",
    })

all_df = pd.DataFrame(rows)
selected_df = all_df[all_df["keep"] == 1].reset_index(drop=True)
dropped_df = all_df[all_df["keep"] == 0].reset_index(drop=True)

summary_df = pd.DataFrame(
    [{
        "n_total": len(all_df),
        "n_kept": len(selected_df),
        "n_dropped": len(dropped_df),
        "n_room_types_kept": int(selected_df["room"].nunique()),
        "n_room_types_dropped": int(dropped_df["room"].nunique()),
    }]
)
kept_counts = (
    selected_df["room"].value_counts().rename_axis("room").reset_index(name="n")
)
drop_counts = (
    dropped_df["room"].value_counts().rename_axis("room").reset_index(name="n")
)

display(summary_df)
display(pd.DataFrame([{"section": "kept room counts"}]))
display(kept_counts)
display(pd.DataFrame([{"section": "dropped room counts"}]))
display(drop_counts)

assert len(selected_df) > 0, "No IRs kept — check EXCLUDE_ROOM_TOKENS"

,n_total,n_kept,n_dropped,n_room_types_kept,n_room_types_dropped
0,270,174,96,51,20


,section
0,kept room counts


,room,n
0,Classroom,35
1,Hallway,17
2,MITCampus,16
3,Bar,14
4,Office,11
5,Bedroom,10
6,Kitchen,5
7,Livingroom,4
8,Restaurant,4
9,Supermarket,4


,section
0,dropped room counts


,room,n
0,Outside,54
1,Bathroom,6
2,Campground,5
3,StreetsOfCambridge,4
4,Train,3
5,ParkingLot,3
6,OfficeBathroom,2
7,StreetsOfcambridge,2
8,StreetsOfBoston,2
9,SubwayStation,2


## Preprocess + save bank

In [3]:
from os import makedirs
import numpy as np
import librosa
import soundfile as sf
import pandas as pd
from IPython.display import display
from tqdm.notebook import tqdm
import warnings

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)


def trim_leading_silence(h: np.ndarray, thresh_db: float) -> tuple[np.ndarray, int]:
    """Drop leading samples below peak * 10^(thresh_db/20). Returns (trimmed, n_dropped)."""
    peak = float(np.max(np.abs(h)))
    if peak <= 0:
        return h, 0
    thr = peak * (10.0 ** (thresh_db / 20.0))
    idx = np.where(np.abs(h) >= thr)[0]
    if len(idx) == 0:
        return h, 0
    start = int(idx[0])
    return h[start:], start


def load_and_prep_ir(path_str: str) -> dict:
    y, sr = librosa.load(path_str, sr=None, mono=True)
    y = y.astype(np.float32)
    n_raw = int(len(y))
    if sr != TARGET_SR:
        y = librosa.resample(y, orig_sr=sr, target_sr=TARGET_SR).astype(np.float32)
    y, n_trim = trim_leading_silence(y, TRIM_LEADING_DB)
    # peak-normalize IR (stable convolution gain)
    peak = float(np.max(np.abs(y)))
    if peak > 0:
        y = (y / peak).astype(np.float32)
    return {
        "ir": y,
        "sr_native": int(sr),
        "n_raw": n_raw,
        "n_trim_leading": int(n_trim),
        "n_samples": int(len(y)),
        "duration_s": float(len(y) / TARGET_SR),
        "peak_pre_norm": peak,
    }


makedirs(FEATURES_DIR, exist_ok=True)

if path.isfile(FEATURES_OUT) and not FORCE_REBUILD:
    data = np.load(FEATURES_OUT, allow_pickle=True)
    skip_df = pd.DataFrame(
        [{
            "status": "skipped_exists",
            "features_out": FEATURES_OUT,
            "n_irs": int(len(data["names"])),
            "target_sr": int(data["target_sr"]) if "target_sr" in data.files else None,
            "bank_tag": str(data["bank_tag"]) if "bank_tag" in data.files else "",
        }]
    )
    display(skip_df)
else:
    irs = []
    names = []
    rooms = []
    n_samples = []
    durations_s = []
    n_trim_leading = []
    sr_native = []
    errors = []

    for row in tqdm(selected_df.to_dict("records"), desc="Prep IRs"):
        try:
            meta = load_and_prep_ir(row["path"])
        except Exception as exc:
            errors.append({"filename": row["filename"], "error": str(exc)})
            continue
        irs.append(meta["ir"])
        names.append(row["filename"])
        rooms.append(row["room"])
        n_samples.append(meta["n_samples"])
        durations_s.append(meta["duration_s"])
        n_trim_leading.append(meta["n_trim_leading"])
        sr_native.append(meta["sr_native"])

    assert irs, "No IRs preprocessed"

    # object array: variable-length float32 waveforms
    ir_obj = np.empty(len(irs), dtype=object)
    for i, arr in enumerate(irs):
        ir_obj[i] = arr

    payload = {
        "irs": ir_obj,
        "names": np.asarray(names),
        "rooms": np.asarray(rooms),
        "n_samples": np.asarray(n_samples, dtype=np.int32),
        "durations_s": np.asarray(durations_s, dtype=np.float32),
        "n_trim_leading": np.asarray(n_trim_leading, dtype=np.int32),
        "sr_native": np.asarray(sr_native, dtype=np.int32),
        "target_sr": np.asarray(TARGET_SR),
        "trim_leading_db": np.asarray(TRIM_LEADING_DB, dtype=np.float32),
        "bank_tag": np.asarray(BANK_TAG),
        "source_dir": np.asarray(IR_SURVEY_DIR),
        "exclude_room_tokens": np.asarray(sorted(EXCLUDE_ROOM_TOKENS)),
    }
    np.savez_compressed(FEATURES_OUT, **payload)

    man = selected_df.copy()
    # align to successfully prepped only
    man = man[man["filename"].isin(names)].reset_index(drop=True)
    meta_map = {n: i for i, n in enumerate(names)}
    man["bank_index"] = man["filename"].map(meta_map)
    man["n_samples"] = man["bank_index"].map(lambda i: n_samples[i])
    man["duration_s"] = man["bank_index"].map(lambda i: durations_s[i])
    man["n_trim_leading"] = man["bank_index"].map(lambda i: n_trim_leading[i])
    man["sr_native"] = man["bank_index"].map(lambda i: sr_native[i])
    man.to_csv(MANIFEST_OUT, index=False)

    save_df = pd.DataFrame(
        [{
            "status": "saved",
            "features_out": FEATURES_OUT,
            "manifest_out": MANIFEST_OUT,
            "n_irs": len(names),
            "duration_s_min": float(np.min(durations_s)),
            "duration_s_median": float(np.median(durations_s)),
            "duration_s_max": float(np.max(durations_s)),
            "n_samples_median": int(np.median(n_samples)),
            "n_errors": len(errors),
        }]
    )
    display(save_df)
    if errors:
        display(pd.DataFrame(errors))

Prep IRs:   0%|          | 0/174 [00:00<?, ?it/s]

,status,features_out,manifest_out,n_irs,duration_s_min,duration_s_median,duration_s_max,n_samples_median,n_errors
0,saved,../../features/rir/ir-survey-indoor_no_bathroo...,../../features/rir/ir-survey-indoor_no_bathroo...,174,0.176708,0.615292,1.992792,29534,0


## Verify

In [4]:
import numpy as np
import pandas as pd
from IPython.display import display

data = np.load(FEATURES_OUT, allow_pickle=True)
irs = data["irs"]
names = data["names"].astype(str)
rooms = data["rooms"].astype(str)
durs = data["durations_s"].astype(np.float32)

assert all(r not in EXCLUDE_ROOM_TOKENS for r in rooms), "Excluded room leaked into bank"
assert all(isinstance(irs[i], np.ndarray) for i in range(len(irs)))
assert all(irs[i].dtype == np.float32 for i in range(len(irs)))
assert int(data["target_sr"]) == TARGET_SR

peaks = [float(np.max(np.abs(irs[i]))) for i in range(len(irs))]
verify_df = pd.DataFrame(
    [{
        "n_irs": len(names),
        "n_rooms": int(len(np.unique(rooms))),
        "target_sr": int(data["target_sr"]),
        "duration_s_min": float(durs.min()),
        "duration_s_median": float(np.median(durs)),
        "duration_s_max": float(durs.max()),
        "peak_min": float(np.min(peaks)),
        "peak_max": float(np.max(peaks)),
        "bank_tag": str(data["bank_tag"]),
    }]
)
display(verify_df)
display(
    pd.Series(rooms)
    .value_counts()
    .rename_axis("room")
    .reset_index(name="n")
    .head(15)
)

,n_irs,n_rooms,target_sr,duration_s_min,duration_s_median,duration_s_max,peak_min,peak_max,bank_tag
0,174,51,48000,0.176708,0.615292,1.992792,1.0,1.0,indoor_no_bathroom


,room,n
0,Classroom,35
1,Hallway,17
2,MITCampus,16
3,Bar,14
4,Office,11
5,Bedroom,10
6,Kitchen,5
7,Livingroom,4
8,Restaurant,4
9,Supermarket,4
